<a href="https://colab.research.google.com/github/epicariello/zone30/blob/main/Zone30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ultimo aggiornamento:** 27 marzo 2026, ore 12:30 UTC  
**Branch:** `claude/analyze-mortality-rates-PI9se`  
**Commit:** Replace utenti_deboli with incidenti_urbani analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


**Questo notebook è stato utilizzato per la realizzazione della tesi L'effetto dell'istituzione della zona 30 km/h sulla mortalità negli incidenti automobilistici**
  

Si carica un dataset ISTAT che contiene gli incidenti con morti e feriti di tutti i comuni d'Italia in serie storica

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

import pandas as pd
import numpy as np
import re

# File path
file_path = '/content/drive/MyDrive/TesiMagistrale/FontiUsate/Incidenti, morti e feriti - comuni.xlsx'

# Carica dati dalla riga 8 (salta 7 righe), NO header
df = pd.read_excel(file_path, skiprows=7, header=None)

# Crea nomi colonne: Comune + morti/feriti/incidenti per 2001-2024
anni = range(2001, 2025)
colonne = ['Comune']
for anno in anni:
    colonne.extend([f'{anno}_morti', f'{anno}_feriti', f'{anno}_incidenti'])

df.columns = colonne

# 🔧 PULIZIA ROBUSTA per risolvere l'errore '..'
def pulisci_valori(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip()
    # Rimuovi '..', ',', e non numeri
    if val_str == '..' or val_str == 'nan':
        return 0
    # Rimuovi virgole per numeri italiani
    val_str = val_str.replace(',', '')
    try:
        return int(float(val_str))
    except (ValueError, TypeError):
        return 0

# Applica pulizia a TUTTE le colonne numeriche
colonne_num = [col for col in df.columns if col != 'Comune']
for col in colonne_num:
    df[col] = df[col].apply(pulisci_valori)

# Pulisci 'Comune'
df['Comune'] = df['Comune'].astype(str).str.strip()
df = df[df['Comune'].str.len() > 0].reset_index(drop=True)

print(f"✅ DataFrame 'df' pronto! Shape: {df.shape}")
print("\nPrime righe:")
print(df[['Comune', '2001_morti', '2001_feriti', '2001_incidenti',
          '2024_morti', '2024_feriti', '2024_incidenti']].head())

print("\n📊 Pronto per analisi!")


✅ DataFrame 'df' pronto! Shape: (8578, 73)

Prime righe:
            Comune  2001_morti  2001_feriti  2001_incidenti  2024_morti  \
0            Agliè           0           10               5           0   
1          Airasca           0           48              25           0   
2     Ala di Stura           0            0               0           0   
3  Albiano d'Ivrea           1            6               5           0   
4  Alice Superiore           4            3               3           0   

   2024_feriti  2024_incidenti  
0            5               2  
1            9               5  
2            2               1  
3            9               4  
4            0               0  

📊 Pronto per analisi!


E' necessario un preprocessing poiché alcuni comuni hanno cambiato provincia nel corso del tempo (in particolare è accaduto a Olbia) per questo i dati si trovano in righe diverse, pur essendo, nel complesso, completo

In [2]:
# ============================================================
# PREPROCESSING - Gestione comuni duplicati
# ============================================================

# --- IDENTIFICAZIONE ---
dupes_mask = df['Comune'].duplicated(keep=False)
dupe_names = df.loc[dupes_mask, 'Comune'].unique()
print(f"Comuni con nome duplicato: {len(dupe_names)}")

colonne_num = [col for col in df.columns if col != 'Comune']

# --- CLASSIFICAZIONE ---
# Complementari: dove una riga ha il dato, l'altra ha 0 (mai sovrapposizione)
#                → merge sicuro per somma (es. Olbia cambio provincia)
# Conflitto:     stessa colonna ha valori > 0 su entrambe le righe
#                → comuni omonimi in province diverse, si lasciano separati

complementary = []
conflict = []

for nome in dupe_names:
    rows = df[df['Comune'] == nome]
    has_conflict = False
    for col in colonne_num:
        vals = rows[col].tolist()
        non_zero = [v for v in vals if v != 0]
        if len(non_zero) > 1:
            has_conflict = True
            break
    if has_conflict:
        conflict.append(nome)
    else:
        complementary.append(nome)

print(f"  → Merge per somma (cambio provincia, es. Olbia): {len(complementary)}")
print(f"  → Lasciati separati (comuni omonimi):            {len(conflict)}")
print(f"  → Comuni omonimi non toccati: {conflict}")

# --- MERGE SOLO PER I COMPLEMENTARI ---
idx_to_drop = set()
merged_rows = []

for nome in complementary:
    rows = df[df['Comune'] == nome]
    idx_to_drop.update(rows.index.tolist())
    merged = {'Comune': nome}
    for col in colonne_num:
        merged[col] = rows[col].sum()
    merged_rows.append(merged)

df_merged = pd.DataFrame(merged_rows)

# --- ASSEMBLAGGIO ---
# I comuni in conflitto rimangono intatti nel df originale
df = df.drop(index=list(idx_to_drop))
df = pd.concat([df, df_merged], ignore_index=True)
df = df.sort_values('Comune').reset_index(drop=True)

print(f"\nDataset pulito: {len(df)} righe, {df['Comune'].nunique()} comuni unici")
print(f"(Nota: {len(conflict)} comuni omonimi mantengono righe doppie)")

# --- VERIFICA Bologna e Olbia ---
for nome in ['Bologna', 'Olbia']:
    row = df[df['Comune'] == nome].iloc[0]
    morti = [str(int(row[f'{a}_morti'])) for a in range(2001, 2025)]
    print(f"\n{nome} - Morti 2001→2024:")
    print(', '.join(morti))

print("\n✅ Preprocessing completato! df aggiornato e pronto per l'analisi.")


Comuni con nome duplicato: 294
  → Merge per somma (cambio provincia, es. Olbia): 288
  → Lasciati separati (comuni omonimi):            6
  → Comuni omonimi non toccati: ['Samone', 'Livo', 'Peglio', 'Castro', 'Valverde', 'San Teodoro']

Dataset pulito: 8223 righe, 8215 comuni unici
(Nota: 6 comuni omonimi mantengono righe doppie)

Bologna - Morti 2001→2024:
32, 39, 46, 35, 28, 36, 28, 20, 26, 28, 20, 22, 7, 18, 25, 16, 15, 25, 18, 14, 12, 23, 21, 11

Olbia - Morti 2001→2024:
6, 10, 15, 9, 11, 4, 8, 1, 6, 7, 5, 3, 10, 2, 4, 3, 3, 6, 2, 3, 3, 3, 4, 6

✅ Preprocessing completato! df aggiornato e pronto per l'analisi.


Si carica un dataset dei Comuni dell'Emilia Romagna

In [3]:
import pandas as pd
import numpy as np

# Carica Comuni Emilia-Romagna
df_comuni_emilia_romagna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Emilia_Romagna.xlsx')

print("✅ df_comuni_emilia_romagna pronto!")
print(f"Shape: {df_comuni_emilia_romagna.shape}")
print("\nColonne:")
print(df_comuni_emilia_romagna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_emilia_romagna.head())

print("\n📊 Riepilogo:")
print(df_comuni_emilia_romagna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_emilia_romagna)}")


✅ df_comuni_emilia_romagna pronto!
Shape: (330, 4)

Colonne:
['Numero', 'Comune', 'Provincia', 'Popolazione (Censimento 2011)']

Prime righe:
   Numero     Comune      Provincia  Popolazione (Censimento 2011)
0       1   Agazzano       Piacenza                           2070
1       2   Albareto          Parma                           2165
2       3    Albinea  Reggio Emilia                           8755
3       4  Alfonsine        Ravenna                          12245
4       5     Alseno       Piacenza                           4823

📊 Riepilogo:
Provincia
Bologna          55
Modena           47
Piacenza         46
Parma            44
Reggio Emilia    42
Forlì-Cesena     30
Rimini           27
Ferrara          21
Ravenna          18
Name: count, dtype: int64

Totale comuni: 330


Si verifica che tutti i comuni dell'Emilia Romagna siano presenti nel dataset iniziale

In [4]:
# Comuni presenti in df_comuni_emilia_romagna ma ASSENTI in df (incidenti)
comuni_emilia_non_incidenti = set(df_comuni_emilia_romagna['Comune'].str.lower()) - set(df['Comune'].str.lower())

print("🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_emilia_non_incidenti)}")
if len(comuni_emilia_non_incidenti) > 0:
    print(list(comuni_emilia_non_incidenti)[:20])  # Primi 20
    if len(comuni_emilia_non_incidenti) > 20:
        print("... e altri", len(comuni_emilia_non_incidenti)-20)
else:
    print("✅ Tutti i comuni Emilia-Romagna sono presenti!")

print("\n📊 Riepilogo:")
print(f"Comuni Emilia-Romagna totali: {len(df_comuni_emilia_romagna)}")
print(f"Comuni con incidenti: {len(set(df['Comune'].str.lower()) & set(df_comuni_emilia_romagna['Comune'].str.lower()))}")
print(f"Copertura: {100*(1-len(comuni_emilia_non_incidenti)/len(df_comuni_emilia_romagna)):.1f}%")


🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Emilia-Romagna sono presenti!

📊 Riepilogo:
Comuni Emilia-Romagna totali: 330
Comuni con incidenti: 330
Copertura: 100.0%


Si carica un dataset contenente tutti i comuni della Sardegna

In [5]:
import pandas as pd
import numpy as np

# Carica Comuni Sardegna
df_comuni_sardegna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Sardegna.xlsx')

print("✅ df_comuni_sardegna pronto!")
print(f"Shape: {df_comuni_sardegna.shape}")
print("\nColonne:")
print(df_comuni_sardegna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_sardegna.head())

print("\n📊 Riepilogo:")
print(df_comuni_sardegna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_sardegna)}")


✅ df_comuni_sardegna pronto!
Shape: (377, 5)

Colonne:
['Comune', 'Provincia', 'Popolazione (2021)', 'Altitudine (m)', 'Superficie (km²)']

Prime righe:
          Comune                  Provincia  Popolazione (2021)  \
0      Abbasanta                   Oristano                2579   
1         Aggius  Gallura Nord-Est Sardegna                1409   
2       Aglientu  Gallura Nord-Est Sardegna                1154   
3   Aidomaggiore                   Oristano                 398   
4  Alà dei Sardi  Gallura Nord-Est Sardegna                1764   

   Altitudine (m)  Superficie (km²)  
0             315             39.85  
1             514             86.31  
2             420            148.19  
3             250             41.21  
4             663            197.99  

📊 Riepilogo:
Provincia
Oristano                     87
Cagliari                     70
Sassari                      66
Nuoro                        53
Medio Campidano              28
Gallura Nord-Est Sardegna    26


Si verifica che tutti i Comuni della Sardegna siano nel dataset degli incidenti

In [6]:
# Verifica comuni SARDAGNA nel df incidenti
comuni_sardegna = set(df_comuni_sardegna['Comune'].str.lower())
comuni_incidenti = set(df['Comune'].str.lower())

comuni_sard_non_incidenti = comuni_sardegna - comuni_incidenti

print("🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_sard_non_incidenti)}")
if len(comuni_sard_non_incidenti) > 0:
    print("\nPrimi 20:")
    for i, comune in enumerate(sorted(comuni_sard_non_incidenti)[:20]):
        print(f"  {i+1}. {comune.title()}")
    if len(comuni_sard_non_incidenti) > 20:
        print(f"\n... e altri {len(comuni_sard_non_incidenti)-20}")
else:
    print("✅ Tutti i comuni Sardegna SONO presenti!")

print(f"\n📊 Copertura:")
print(f"Comuni Sardegna totali: {len(df_comuni_sardegna):,}")
print(f"Comuni con incidenti: {len(comuni_sardegna & comuni_incidenti):,}")
print(f"Copertura: {100*(len(comuni_sardegna & comuni_incidenti)/len(df_comuni_sardegna)):.1f}%")


🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Sardegna SONO presenti!

📊 Copertura:
Comuni Sardegna totali: 377
Comuni con incidenti: 377
Copertura: 100.0%


In [7]:

import pandas as pd

# Carica il CSV della popolazione per comune per anno
df_pop = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/popolazione_comuni_per_anno.csv',
    sep=';',
    decimal=',',
    encoding='latin-1'
)

# Rinomina per chiarezza
df_pop = df_pop.rename(columns={'codice': 'Codice', 'comune': 'Comune'})

print("✅ df_pop pronto!")
print(f"Shape: {df_pop.shape}")
print("\nColonne:")
print(df_pop.columns.tolist())
print("\nPrime righe:")
print(df_pop.head())

# -------------------------------------------------------
# Verifica: tutti i comuni di df sono presenti in df_pop?
# -------------------------------------------------------
comuni_df = set(df['Comune'].str.strip().str.lower())
comuni_pop = set(df_pop['Comune'].str.strip().str.lower())

assenti_in_pop = comuni_df - comuni_pop

print("\n🔍 Comuni presenti in df (incidenti) ma ASSENTI in df_pop (popolazione):")
print(f"Totale: {len(assenti_in_pop)}")
if assenti_in_pop:
    for c in sorted(assenti_in_pop):
        print(f"  - {c.title()}")
else:
    print("✅ Tutti i comuni di df sono presenti in df_pop!")

print(f"\n📊 Riepilogo copertura:")
print(f"Comuni in df (incidenti):   {len(comuni_df):,}")
print(f"Comuni in df_pop:            {len(comuni_pop):,}")
print(f"Comuni coperti:              {len(comuni_df & comuni_pop):,}")
print(f"Copertura: {100 * len(comuni_df & comuni_pop) / len(comuni_df):.1f}%")


✅ df_pop pronto!
Shape: (7991, 26)

Colonne:
['Codice', 'Comune', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Prime righe:
   Codice           Comune    2002    2003    2004    2005    2006    2007  \
0    1001            Agliè  2557.0  2538.0  2588.0  2679.0  2674.0  2662.0   
1    1002          Airasca  3543.0  3544.0  3605.0  3648.0  3642.0  3666.0   
2    1003     Ala di Stura   480.0   468.0   461.0   458.0   458.0   467.0   
3    1004  Albiano d'Ivrea  1687.0  1673.0  1702.0  1696.0  1687.0  1661.0   
4    1006           Almese  5648.0  5679.0  5805.0  5863.0  6134.0  6157.0   

     2008    2009  ...    2016    2017    2018    2019    2020    2021  \
0  2658.0  2638.0  ...  2644.0  2661.0  2658.0  2634.0  2621.0  2545.0   
1  3775.0  3794.0  ...  3761.0  3726.0  3671.0  3613.0  3598.0  3633.0   
2   480.0   477.0  ...   464.0   460.

In [ ]:
import unicodedata
import pandas as pd
import numpy as np

def normalizza(s):
    """Rimuove accenti e apostrofi, lowercase — per join robusto sui nomi comuni."""
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", "").replace("`", "")

anni_urban = list(range(2010, 2025))

# Caricamento incidenti stradali in ambito urbano
df_urban = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/incidenti_urbani.csv',
    sep=';', encoding='utf-8'
)
df_urban = df_urban.rename(columns={'nome_comune': 'Comune'})
df_urban['_key'] = df_urban['Comune'].apply(normalizza)

# Join con df_pop — chiave normalizzata su entrambi i lati
_dp_urban = df_pop.copy()
_dp_urban['_key'] = _dp_urban['Comune'].apply(normalizza)
pop_cols = ['_key'] + [str(a) for a in anni_urban if str(a) in _dp_urban.columns]
_dj_urban = df_urban.merge(_dp_urban[pop_cols], on='_key', how='left')

n_no_pop = _dj_urban[str(anni_urban[0])].isna().sum()
print(f"✅ incidenti_urbani caricato: {len(_dj_urban):,} comuni, anni 2010–2024")
print(f"   Join popolazione: {len(_dj_urban) - n_no_pop:,} comuni con pop / {n_no_pop} senza match")
if n_no_pop > 0:
    mask_no = _dj_urban[str(anni_urban[0])].isna()
    print("   Esempi senza match:", _dj_urban.loc[mask_no, 'Comune'].head(10).tolist())


In [ ]:
# Calcolo tassi per singolo comune (per 1.000 ab.)
for anno in anni_urban:
    pop = pd.to_numeric(_dj_urban[str(anno)], errors='coerce').replace(0, np.nan)
    for m in ('morti', 'feriti', 'incidenti'):
        cnt = pd.to_numeric(_dj_urban[f'{anno}_{m}'], errors='coerce')
        _dj_urban[f'{anno}_tasso_{m}_urban'] = (cnt / pop * 1000).round(4)

# Controllo: comuni dove TUTTI i tassi sono NaN (join pop fallito)
tasso_check = [f'{a}_tasso_morti_urban' for a in anni_urban]
no_tasso = _dj_urban[tasso_check].isna().all(axis=1)
print(f"✅ Tassi incidenti urbani calcolati. Shape _dj_urban: {_dj_urban.shape}")
print(f"⚠️  Comuni senza tassi (join pop fallito): {no_tasso.sum():,} su {len(_dj_urban):,}")
if no_tasso.sum() > 0:
    print(_dj_urban.loc[no_tasso, ['Comune', '_key']].head(10).to_string(index=False))


In [8]:
import pandas as pd
import numpy as np

# -------------------------------------------------------
# Costruzione dataset definitivo
# -------------------------------------------------------

# Chiave di join case-insensitive (colonna temporanea)
df_tmp = df.copy()
df_tmp['_key'] = df_tmp['Comune'].str.strip().str.lower()

df_pop_tmp = df_pop.copy()
df_pop_tmp['_key'] = df_pop_tmp['Comune'].str.strip().str.lower()

# Inner join → solo comuni presenti in entrambi
df_final = df_tmp.merge(df_pop_tmp, on='_key', suffixes=('', '_pop'))

# Rimuovi colonne ausiliarie
df_final = df_final.drop(columns=['_key', 'Comune_pop'])

# -------------------------------------------------------
# Calcolo tassi per gli anni sovrapposti: 2002–2024
# (df_pop copre 2002-2025, df copre 2001-2024)
# -------------------------------------------------------
anni_comuni = range(2002, 2025)

for anno in anni_comuni:
    pop = df_final[str(anno)].replace(0, np.nan)
    df_final[f'{anno}_tasso_mortalita']  = (df_final[f'{anno}_morti']  / pop * 1000).round(4)
    df_final[f'{anno}_tasso_ferimento']  = (df_final[f'{anno}_feriti'] / pop * 1000).round(4)

# Rimuovi le colonne grezze di popolazione (2002-2025)
pop_year_cols = [str(a) for a in range(2002, 2026) if str(a) in df_final.columns]
df_final = df_final.drop(columns=pop_year_cols)

# Rimuovi anno 2001 (no dati popolazione → niente tassi) e 2025 (no dati incidenti)
cols_2001 = [c for c in df_final.columns if c.startswith('2001_')]
cols_2025 = [c for c in df_final.columns if c.startswith('2025_') or c == '2025']
df_final = df_final.drop(columns=cols_2001 + cols_2025, errors='ignore')

# Rimuovi Codice
df_final = df_final.drop(columns=['Codice'], errors='ignore')

df_final = df_final.reset_index(drop=True)

print("✅ df_final pronto!")
print(f"Shape: {df_final.shape}")
print(f"Comuni: {len(df_final):,}")
print(f"Anni coperti: 2002–2024")

print("\nColonne (prime e ultime):")
cols = df_final.columns.tolist()
print(cols[:4], "...", cols[-6:])

print("\nEsempio — Olbia:")
esempio = df_final[df_final['Comune'] == 'Olbia']
if not esempio.empty:
    for anno in [2002, 2010, 2024]:
        row = esempio.iloc[0]
        print(f"  {anno}: morti={row[f'{anno}_morti']}, feriti={row[f'{anno}_feriti']}, "
              f"tasso_mortalita={row[f'{anno}_tasso_mortalita']}, "
              f"tasso_ferimento={row[f'{anno}_tasso_ferimento']}")

print(f"\n📊 Comuni rimossi (assenti in df_pop): "
      f"{len(df) - len(df_final)} su {len(df)}")

✅ df_final pronto!
Shape: (7993, 116)
Comuni: 7,993
Anni coperti: 2002–2024

Colonne (prime e ultime):
['Comune', '2002_morti', '2002_feriti', '2002_incidenti'] ... ['2022_tasso_mortalita', '2022_tasso_ferimento', '2023_tasso_mortalita', '2023_tasso_ferimento', '2024_tasso_mortalita', '2024_tasso_ferimento']

Esempio — Olbia:
  2002: morti=10, feriti=521, tasso_mortalita=0.2204, tasso_ferimento=11.4841
  2010: morti=7, feriti=466, tasso_mortalita=0.1315, tasso_ferimento=8.7554
  2024: morti=6, feriti=381, tasso_mortalita=0.0976, tasso_ferimento=6.197

📊 Comuni rimossi (assenti in df_pop): 230 su 8223


In [16]:
print(df_final .columns.tolist())

['Comune', '2002_morti', '2002_feriti', '2002_incidenti', '2003_morti', '2003_feriti', '2003_incidenti', '2004_morti', '2004_feriti', '2004_incidenti', '2005_morti', '2005_feriti', '2005_incidenti', '2006_morti', '2006_feriti', '2006_incidenti', '2007_morti', '2007_feriti', '2007_incidenti', '2008_morti', '2008_feriti', '2008_incidenti', '2009_morti', '2009_feriti', '2009_incidenti', '2010_morti', '2010_feriti', '2010_incidenti', '2011_morti', '2011_feriti', '2011_incidenti', '2012_morti', '2012_feriti', '2012_incidenti', '2013_morti', '2013_feriti', '2013_incidenti', '2014_morti', '2014_feriti', '2014_incidenti', '2015_morti', '2015_feriti', '2015_incidenti', '2016_morti', '2016_feriti', '2016_incidenti', '2017_morti', '2017_feriti', '2017_incidenti', '2018_morti', '2018_feriti', '2018_incidenti', '2019_morti', '2019_feriti', '2019_incidenti', '2020_morti', '2020_feriti', '2020_incidenti', '2021_morti', '2021_feriti', '2021_incidenti', '2022_morti', '2022_feriti', '2022_incidenti', '2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

anni = list(range(2002, 2025))

df_joined = df.copy()
df_joined['_key'] = df_joined['Comune'].str.strip().str.lower()
df_pop_tmp2 = df_pop.copy()
df_pop_tmp2['_key'] = df_pop_tmp2['Comune'].str.strip().str.lower()
df_joined = df_joined.merge(df_pop_tmp2[['_key'] + [str(a) for a in anni]], on='_key')

is_bologna = df_joined['_key'] == 'bologna'

def tasso_aggregato(mask, anni, colonna):
    tassi = []
    for anno in anni:
        val = df_joined.loc[mask, f'{anno}_{colonna}'].sum()
        pop = df_joined.loc[mask, str(anno)].replace(0, np.nan).sum()
        tassi.append(val / pop * 1000 if pop else np.nan)
    return tassi

tasso_mort_it  = tasso_aggregato(~is_bologna, anni, 'morti')
tasso_mort_bo  = tasso_aggregato( is_bologna, anni, 'morti')
tasso_ferim_it = tasso_aggregato(~is_bologna, anni, 'feriti')
tasso_ferim_bo = tasso_aggregato( is_bologna, anni, 'feriti')
tasso_incid_it = tasso_aggregato(~is_bologna, anni, 'incidenti')
tasso_incid_bo = tasso_aggregato( is_bologna, anni, 'incidenti')

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_mort_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni, tasso_mort_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Tasso Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Morti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Ferimento — Bologna vs Italia ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_ferim_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni, tasso_ferim_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Tasso Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Feriti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Incidenti — Bologna vs Italia ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_incid_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni, tasso_incid_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Tasso Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Incidenti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

# ---- Riepilogo numerico ----
print(f"{'Anno':6} | {'Mort.IT':>8} | {'Mort.BO':>8} | {'Ferit.IT':>9} | {'Ferit.BO':>9} | {'Incid.IT':>9} | {'Incid.BO':>9}")
print("-" * 80)
for i, anno in enumerate(anni):
    print(f"{anno:<6} | {tasso_mort_it[i]:>8.4f} | {tasso_mort_bo[i]:>8.4f} | "
          f"{tasso_ferim_it[i]:>9.4f} | {tasso_ferim_bo[i]:>9.4f} | "
          f"{tasso_incid_it[i]:>9.4f} | {tasso_incid_bo[i]:>9.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

anni = list(range(2002, 2025))

df_joined = df.copy()
df_joined['_key'] = df_joined['Comune'].str.strip().str.lower()
df_pop_tmp2 = df_pop.copy()
df_pop_tmp2['_key'] = df_pop_tmp2['Comune'].str.strip().str.lower()
df_joined = df_joined.merge(df_pop_tmp2[['_key'] + [str(a) for a in anni]], on='_key')

er_keys     = set(df_comuni_emilia_romagna['Comune'].str.strip().str.lower())
is_bologna  = df_joined['_key'] == 'bologna'
is_er_no_bo = df_joined['_key'].isin(er_keys) & ~is_bologna

def tasso_aggregato(mask, anni, colonna):
    tassi = []
    for anno in anni:
        val = df_joined.loc[mask, f'{anno}_{colonna}'].sum()
        pop = df_joined.loc[mask, str(anno)].replace(0, np.nan).sum()
        tassi.append(val / pop * 1000 if pop else np.nan)
    return tassi

tasso_mort_er  = tasso_aggregato(is_er_no_bo, anni, 'morti')
tasso_mort_bo  = tasso_aggregato(is_bologna,  anni, 'morti')
tasso_ferim_er = tasso_aggregato(is_er_no_bo, anni, 'feriti')
tasso_ferim_bo = tasso_aggregato(is_bologna,  anni, 'feriti')
tasso_incid_er = tasso_aggregato(is_er_no_bo, anni, 'incidenti')
tasso_incid_bo = tasso_aggregato(is_bologna,  anni, 'incidenti')
print(f'Comuni Emilia-Romagna (esclusa Bologna) nel dataset: {is_er_no_bo.sum()}')

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_mort_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni, tasso_mort_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Tasso Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Morti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Ferimento — Bologna vs Emilia-Romagna ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_ferim_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni, tasso_ferim_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Tasso Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Feriti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Incidenti — Bologna vs Emilia-Romagna ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_incid_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni, tasso_incid_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Tasso Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Incidenti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

# ---- Riepilogo numerico ----
print(f"{'Anno':6} | {'Mort.ER':>8} | {'Mort.BO':>8} | {'Ferit.ER':>9} | {'Ferit.BO':>9} | {'Incid.ER':>9} | {'Incid.BO':>9}")
print("-" * 80)
for i, anno in enumerate(anni):
    print(f"{anno:<6} | {tasso_mort_er[i]:>8.4f} | {tasso_mort_bo[i]:>8.4f} | "
          f"{tasso_ferim_er[i]:>9.4f} | {tasso_ferim_bo[i]:>9.4f} | "
          f"{tasso_incid_er[i]:>9.4f} | {tasso_incid_bo[i]:>9.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

anni = list(range(2002, 2025))

_dj = df.copy()
_dj['_key'] = _dj['Comune'].str.strip().str.lower()
_dp = df_pop.copy()
_dp['_key'] = _dp['Comune'].str.strip().str.lower()
_dj = _dj.merge(_dp[['_key'] + [str(a) for a in anni]], on='_key')

is_olbia       = _dj['_key'] == 'olbia'
is_it_no_olbia = ~is_olbia

def _tasso(mask, anni, colonna):
    tassi = []
    for anno in anni:
        val = _dj.loc[mask, f'{anno}_{colonna}'].sum()
        pop = _dj.loc[mask, str(anno)].replace(0, np.nan).sum()
        tassi.append(val / pop * 1000 if pop else np.nan)
    return tassi

tasso_mort_it  = _tasso(is_it_no_olbia, anni, 'morti')
tasso_mort_ol  = _tasso(is_olbia,       anni, 'morti')
tasso_ferim_it = _tasso(is_it_no_olbia, anni, 'feriti')
tasso_ferim_ol = _tasso(is_olbia,       anni, 'feriti')
tasso_incid_it = _tasso(is_it_no_olbia, anni, 'incidenti')
tasso_incid_ol = _tasso(is_olbia,       anni, 'incidenti')

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_mort_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni, tasso_mort_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Tasso Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Morti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Ferimento — Olbia vs Italia ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_ferim_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni, tasso_ferim_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Tasso Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Feriti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Incidenti — Olbia vs Italia ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_incid_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni, tasso_incid_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Tasso Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Incidenti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

# ---- Riepilogo numerico ----
print(f"{'Anno':6} | {'Mort.IT':>8} | {'Mort.OL':>8} | {'Ferit.IT':>9} | {'Ferit.OL':>9} | {'Incid.IT':>9} | {'Incid.OL':>9}")
print("-" * 80)
for i, anno in enumerate(anni):
    print(f"{anno:<6} | {tasso_mort_it[i]:>8.4f} | {tasso_mort_ol[i]:>8.4f} | "
          f"{tasso_ferim_it[i]:>9.4f} | {tasso_ferim_ol[i]:>9.4f} | "
          f"{tasso_incid_it[i]:>9.4f} | {tasso_incid_ol[i]:>9.4f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

anni = list(range(2002, 2025))

_dj = df.copy()
_dj['_key'] = _dj['Comune'].str.strip().str.lower()
_dp = df_pop.copy()
_dp['_key'] = _dp['Comune'].str.strip().str.lower()
_dj = _dj.merge(_dp[['_key'] + [str(a) for a in anni]], on='_key')

sard_keys     = set(df_comuni_sardegna['Comune'].str.strip().str.lower())
is_olbia      = _dj['_key'] == 'olbia'
is_sard_no_ol = _dj['_key'].isin(sard_keys) & ~is_olbia

def _tasso(mask, anni, colonna):
    tassi = []
    for anno in anni:
        val = _dj.loc[mask, f'{anno}_{colonna}'].sum()
        pop = _dj.loc[mask, str(anno)].replace(0, np.nan).sum()
        tassi.append(val / pop * 1000 if pop else np.nan)
    return tassi

tasso_mort_sa  = _tasso(is_sard_no_ol, anni, 'morti')
tasso_mort_ol  = _tasso(is_olbia,      anni, 'morti')
tasso_ferim_sa = _tasso(is_sard_no_ol, anni, 'feriti')
tasso_ferim_ol = _tasso(is_olbia,      anni, 'feriti')
tasso_incid_sa = _tasso(is_sard_no_ol, anni, 'incidenti')
tasso_incid_ol = _tasso(is_olbia,      anni, 'incidenti')
print(f'Comuni Sardegna (esclusa Olbia) nel dataset: {is_sard_no_ol.sum()}')

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_mort_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni, tasso_mort_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Tasso Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Morti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Ferimento — Olbia vs Sardegna ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_ferim_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni, tasso_ferim_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Tasso Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Feriti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Grafico Incidenti — Olbia vs Sardegna ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni, tasso_incid_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni, tasso_incid_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Tasso Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9); ax.set_ylabel('Incidenti per 1.000 ab.', fontsize=9)
ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

# ---- Riepilogo numerico ----
print(f"{'Anno':6} | {'Mort.SA':>8} | {'Mort.OL':>8} | {'Ferit.SA':>9} | {'Ferit.OL':>9} | {'Incid.SA':>9} | {'Incid.OL':>9}")
print("-" * 80)
for i, anno in enumerate(anni):
    print(f"{anno:<6} | {tasso_mort_sa[i]:>8.4f} | {tasso_mort_ol[i]:>8.4f} | "
          f"{tasso_ferim_sa[i]:>9.4f} | {tasso_ferim_ol[i]:>9.4f} | "
          f"{tasso_incid_sa[i]:>9.4f} | {tasso_incid_ol[i]:>9.4f}")


## Incidenti Urbani — Analisi Descrittiva

Confronto dei tassi di mortalità, ferimento e incidenti **in ambito urbano** per 1.000 abitanti (2010–2024).

In [ ]:
def tasso_agg_urban(mask, colonna):
    tassi = []
    for anno in anni_urban:
        val = pd.to_numeric(_dj_urban.loc[mask, f'{anno}_{colonna}'], errors='coerce').sum()
        pop = pd.to_numeric(_dj_urban.loc[mask, str(anno)], errors='coerce').replace(0, float('nan')).sum()
        tassi.append(val / pop * 1000 if pop else float('nan'))
    return tassi

is_bo_u     = _dj_urban['_key'] == normalizza('bologna')
ref_it_bo_u = ~is_bo_u

tu_mort_it = tasso_agg_urban(ref_it_bo_u, 'morti')
tu_mort_bo = tasso_agg_urban(is_bo_u,     'morti')
tu_fer_it  = tasso_agg_urban(ref_it_bo_u, 'feriti')
tu_fer_bo  = tasso_agg_urban(is_bo_u,     'feriti')
tu_inc_it  = tasso_agg_urban(ref_it_bo_u, 'incidenti')
tu_inc_bo  = tasso_agg_urban(is_bo_u,     'incidenti')
print('Dati calcolati — Bologna vs Italia (incidenti urbani)')

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_mort_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni_urban, tu_mort_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Incidenti Urbani: Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Morti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Ferimento ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_fer_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni_urban, tu_fer_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Incidenti Urbani: Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Feriti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Incidenti ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_inc_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Bologna)')
ax.plot(anni_urban, tu_inc_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Italia (esclusa Bologna) vs Bologna — Incidenti Urbani: Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Incidenti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Anno':6} | {'Mort.IT':>8} | {'Mort.BO':>8} | {'Fer.IT':>8} | {'Fer.BO':>8} | {'Inc.IT':>8} | {'Inc.BO':>8}")
print("-" * 75)
for i, anno in enumerate(anni_urban):
    print(f"{anno:<6} | {tu_mort_it[i]:>8.4f} | {tu_mort_bo[i]:>8.4f} | "
          f"{tu_fer_it[i]:>8.4f} | {tu_fer_bo[i]:>8.4f} | "
          f"{tu_inc_it[i]:>8.4f} | {tu_inc_bo[i]:>8.4f}")


In [ ]:
er_keys_u   = {normalizza(k) for k in df_comuni_emilia_romagna['Comune'].str.strip()}
is_bo_u     = _dj_urban['_key'] == normalizza('bologna')
ref_er_bo_u = _dj_urban['_key'].isin(er_keys_u) & ~is_bo_u

tu_mort_er = tasso_agg_urban(ref_er_bo_u, 'morti')
tu_mort_bo = tasso_agg_urban(is_bo_u,     'morti')
tu_fer_er  = tasso_agg_urban(ref_er_bo_u, 'feriti')
tu_fer_bo  = tasso_agg_urban(is_bo_u,     'feriti')
tu_inc_er  = tasso_agg_urban(ref_er_bo_u, 'incidenti')
tu_inc_bo  = tasso_agg_urban(is_bo_u,     'incidenti')
print(f'ER comuni trovati in _dj_urban: {ref_er_bo_u.sum()}')

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_mort_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni_urban, tu_mort_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Incidenti Urbani: Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Morti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Ferimento ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_fer_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni_urban, tu_fer_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Incidenti Urbani: Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Feriti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Incidenti ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_inc_er, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Emilia-Romagna (esclusa Bologna)')
ax.plot(anni_urban, tu_inc_bo, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Bologna')
ax.set_title('Emilia-Romagna (esclusa Bologna) vs Bologna — Incidenti Urbani: Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Incidenti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2024, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (2024)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Anno':6} | {'Mort.ER':>8} | {'Mort.BO':>8} | {'Fer.ER':>8} | {'Fer.BO':>8} | {'Inc.ER':>8} | {'Inc.BO':>8}")
print("-" * 75)
for i, anno in enumerate(anni_urban):
    print(f"{anno:<6} | {tu_mort_er[i]:>8.4f} | {tu_mort_bo[i]:>8.4f} | "
          f"{tu_fer_er[i]:>8.4f} | {tu_fer_bo[i]:>8.4f} | "
          f"{tu_inc_er[i]:>8.4f} | {tu_inc_bo[i]:>8.4f}")


In [ ]:
is_ol_u     = _dj_urban['_key'] == normalizza('olbia')
ref_it_ol_u = ~is_ol_u

tu_mort_it = tasso_agg_urban(ref_it_ol_u, 'morti')
tu_mort_ol = tasso_agg_urban(is_ol_u,     'morti')
tu_fer_it  = tasso_agg_urban(ref_it_ol_u, 'feriti')
tu_fer_ol  = tasso_agg_urban(is_ol_u,     'feriti')
tu_inc_it  = tasso_agg_urban(ref_it_ol_u, 'incidenti')
tu_inc_ol  = tasso_agg_urban(is_ol_u,     'incidenti')
print('Dati calcolati — Olbia vs Italia (incidenti urbani)')

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_mort_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni_urban, tu_mort_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Incidenti Urbani: Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Morti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Ferimento ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_fer_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni_urban, tu_fer_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Incidenti Urbani: Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Feriti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Incidenti ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_inc_it, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Italia (esclusa Olbia)')
ax.plot(anni_urban, tu_inc_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Italia (esclusa Olbia) vs Olbia — Incidenti Urbani: Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Incidenti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Anno':6} | {'Mort.IT':>8} | {'Mort.OL':>8} | {'Fer.IT':>8} | {'Fer.OL':>8} | {'Inc.IT':>8} | {'Inc.OL':>8}")
print("-" * 75)
for i, anno in enumerate(anni_urban):
    print(f"{anno:<6} | {tu_mort_it[i]:>8.4f} | {tu_mort_ol[i]:>8.4f} | "
          f"{tu_fer_it[i]:>8.4f} | {tu_fer_ol[i]:>8.4f} | "
          f"{tu_inc_it[i]:>8.4f} | {tu_inc_ol[i]:>8.4f}")


In [ ]:
sard_keys_u = {normalizza(k) for k in df_comuni_sardegna['Comune'].str.strip()}
is_ol_u     = _dj_urban['_key'] == normalizza('olbia')
ref_sa_ol_u = _dj_urban['_key'].isin(sard_keys_u) & ~is_ol_u

tu_mort_sa = tasso_agg_urban(ref_sa_ol_u, 'morti')
tu_mort_ol = tasso_agg_urban(is_ol_u,     'morti')
tu_fer_sa  = tasso_agg_urban(ref_sa_ol_u, 'feriti')
tu_fer_ol  = tasso_agg_urban(is_ol_u,     'feriti')
tu_inc_sa  = tasso_agg_urban(ref_sa_ol_u, 'incidenti')
tu_inc_ol  = tasso_agg_urban(is_ol_u,     'incidenti')
print(f'SA comuni trovati in _dj_urban: {ref_sa_ol_u.sum()}')

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Mortalità ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_mort_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni_urban, tu_mort_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Incidenti Urbani: Mortalità', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Morti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Ferimento ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_fer_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni_urban, tu_fer_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Incidenti Urbani: Ferimento', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Feriti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ---- Grafico Incidenti ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anni_urban, tu_inc_sa, marker='o', linewidth=2, markersize=4,
        color='steelblue', label='Sardegna (esclusa Olbia)')
ax.plot(anni_urban, tu_inc_ol, marker='s', linewidth=2, markersize=5,
        color='tomato', label='Olbia')
ax.set_title('Sardegna (esclusa Olbia) vs Olbia — Incidenti Urbani: Incidenti', fontsize=11, fontweight='bold')
ax.set_xlabel('Anno', fontsize=9)
ax.set_ylabel('Incidenti urbani per 1.000 ab.', fontsize=9)
ax.set_xticks(anni_urban); ax.set_xticklabels(anni_urban, rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.axvline(2021.5, color='green', linestyle='--', linewidth=1.5, label='Zona 30 (gen 2022)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.5); ax.grid(axis='x', linestyle=':', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Anno':6} | {'Mort.SA':>8} | {'Mort.OL':>8} | {'Fer.SA':>8} | {'Fer.OL':>8} | {'Inc.SA':>8} | {'Inc.OL':>8}")
print("-" * 75)
for i, anno in enumerate(anni_urban):
    print(f"{anno:<6} | {tu_mort_sa[i]:>8.4f} | {tu_mort_ol[i]:>8.4f} | "
          f"{tu_fer_sa[i]:>8.4f} | {tu_fer_ol[i]:>8.4f} | "
          f"{tu_inc_sa[i]:>8.4f} | {tu_inc_ol[i]:>8.4f}")


## DiD Standard — Stima OLS con effetti fissi anno

Il modello stimato è:

$$Y_{it} = \\alpha + \\beta \\cdot \\text{Trattato}_i + \\sum_{t \\neq t_0} \\gamma_t \\cdot \\mathbf{1}[\\text{anno}=t] + \\delta \\cdot (\\text{Trattato}_i \\times \\text{Post}_t) + \\varepsilon_{it}$$

dove:
- $\\text{Trattato}_i = 1$ per la città trattata (Bologna / Olbia), 0 per il controllo aggregato  
- $\\text{Post}_t = 1$ per gli anni $\\geq$ anno di introduzione della Zona 30  
- $C(\\text{anno})$ sono effetti fissi anno (dummy per ogni anno, escluso il riferimento = ultimo anno pre-trattamento)  
- $\\delta$ è il coefficiente DiD (ATT — Average Treatment effect on the Treated)  
- Errori standard robusti **HC1**

**Anno di trattamento**: Bologna → 2024, Olbia → 2022

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

anni = list(range(2002, 2025))

# -------------------------------------------------------
# Ricostruzione df_joined (sicurezza esecuzione isolata)
# -------------------------------------------------------
_dj = df.copy()
_dj['_key'] = _dj['Comune'].str.strip().str.lower()
_dp = df_pop.copy()
_dp['_key'] = _dp['Comune'].str.strip().str.lower()
_dj = _dj.merge(_dp[['_key'] + [str(a) for a in anni]], on='_key')

er_keys   = set(df_comuni_emilia_romagna['Comune'].str.strip().str.lower())
sard_keys = set(df_comuni_sardegna['Comune'].str.strip().str.lower())

# -------------------------------------------------------
# Funzioni helper
# -------------------------------------------------------
def _tasso_agg(mask, colonna):
    """Tasso aggregato per 1.000 ab., per ogni anno."""
    tassi = []
    for anno in anni:
        val = _dj.loc[mask, f'{anno}_{colonna}'].sum()
        pop = _dj.loc[mask, str(anno)].replace(0, np.nan).sum()
        tassi.append(val / pop * 1000 if pop else np.nan)
    return tassi

def build_panel(tasso_ctrl, tasso_tratt, anno_tratt, label_ctrl, label_tratt):
    """Dataset panel long-format per DiD."""
    rows = []
    for i, a in enumerate(anni):
        rows.append({'anno': a, 'tasso': tasso_ctrl[i],  'trattato': 0, 'gruppo': label_ctrl})
        rows.append({'anno': a, 'tasso': tasso_tratt[i], 'trattato': 1, 'gruppo': label_tratt})
    panel = pd.DataFrame(rows)
    panel['post'] = (panel['anno'] >= anno_tratt).astype(int)
    panel['did']  = panel['trattato'] * panel['post']
    return panel

def stima_did(panel, anno_tratt):
    """OLS DiD con effetti fissi anno (HC1). Restituisce (result, anno_ref)."""
    anno_ref = anno_tratt - 1          # ultimo anno pre-trattamento come riferimento
    formula  = f'tasso ~ trattato + did + C(anno, Treatment({anno_ref}))'
    res = smf.ols(formula, data=panel.dropna(subset=['tasso'])).fit(cov_type='HC1')
    return res, anno_ref

def _stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''

def extract_did(res, label):
    """Estrae il coefficiente DiD e diagnostiche."""
    c  = res.params['did']
    se = res.bse['did']
    t  = res.tvalues['did']
    p  = res.pvalues['did']
    ci = res.conf_int().loc['did']
    return {
        'label'  : label,
        'coef'   : c,
        'se'     : se,
        'tstat'  : t,
        'pval'   : p,
        'ci_low' : ci.iloc[0],
        'ci_high': ci.iloc[1],
        'stars'  : _stars(p),
        'N'      : int(res.nobs),
        'R2'     : res.rsquared,
    }

def print_did(r):
    print(f"  δ = {r['coef']:+.5f}{r['stars']}")
    print(f"  SE (HC1) = {r['se']:.5f}   t = {r['tstat']:+.3f}   p = {r['pval']:.4f}")
    print(f"  IC 95%   = [{r['ci_low']:+.5f}, {r['ci_high']:+.5f}]")
    print(f"  N obs = {r['N']}   R² = {r['R2']:.4f}")

print("✅ Funzioni DiD pronte.")


In [ ]:
# ====================================================
# DiD — Bologna  (Zona 30: 2024)
# ====================================================
ANNO_TRATT_BO = 2024

is_bo = _dj['_key'] == 'bologna'

# Tassi aggregati — mortalità e ferimento
t_bo        = {c: _tasso_agg(is_bo,                                   c) for c in ('morti', 'feriti', 'incidenti')}
t_it_no_bo  = {c: _tasso_agg(~is_bo,                                  c) for c in ('morti', 'feriti', 'incidenti')}
t_er_no_bo  = {c: _tasso_agg(_dj['_key'].isin(er_keys) & ~is_bo,      c) for c in ('morti', 'feriti', 'incidenti')}

OUTCOMES = [('morti',     'Mortalità', 'Morti per 1.000 ab.'),
            ('feriti',    'Ferimento', 'Feriti per 1.000 ab.'),
            ('incidenti', 'Incidenti', 'Incidenti per 1.000 ab.')]

_did_results = []

for col, titolo_out, ylabel in OUTCOMES:
    panel_it = build_panel(t_it_no_bo[col], t_bo[col], ANNO_TRATT_BO,
                           'Italia (excl. BO)', 'Bologna')
    panel_er = build_panel(t_er_no_bo[col], t_bo[col], ANNO_TRATT_BO,
                           'Emilia-Romagna (excl. BO)', 'Bologna')

    res_it, ref_it = stima_did(panel_it, ANNO_TRATT_BO)
    res_er, ref_er = stima_did(panel_er, ANNO_TRATT_BO)

    r_it = extract_did(res_it, f'Bologna vs Italia — {titolo_out}')
    r_er = extract_did(res_er, f'Bologna vs Emilia-Romagna — {titolo_out}')
    r_it['outcome'] = r_er['outcome'] = col
    _did_results += [r_it, r_er]

    # ---- Output numerico ----
    print('=' * 66)
    print(f'DiD — Bologna (Post ≥ {ANNO_TRATT_BO})   [{titolo_out} — tasso per 1.000 ab.]')
    print('=' * 66)
    print(f'\n  vs Italia (anno ref = {ref_it})')
    print_did(r_it)
    print(f'\n  vs Emilia-Romagna (anno ref = {ref_er})')
    print_did(r_er)
    print()

print('Significatività: *** p<0.01  ** p<0.05  * p<0.1')

# ---- Grafici parallel trends (2×2: outcome × gruppo controllo) ----
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
fig.suptitle('Bologna — Parallel Trends (tassi per 1.000 ab.)',
             fontsize=12, fontweight='bold')

idx = 0
for row_ax, (col, titolo_out, ylabel) in zip(axes, OUTCOMES):
    for ax, ctrl_tasso, lbl_ctrl in zip(
            row_ax,
            [t_it_no_bo[col], t_er_no_bo[col]],
            ['Italia (esclusa Bologna)', 'Emilia-Romagna (esclusa Bologna)']):

        r = _did_results[idx]; idx += 1
        ax.plot(anni, ctrl_tasso,  'o-', color='steelblue', lw=2, ms=4, label=lbl_ctrl)
        ax.plot(anni, t_bo[col],   's-', color='tomato',    lw=2, ms=5, label='Bologna')
        ax.axvline(ANNO_TRATT_BO - 0.5, color='green', ls='--', lw=1.5,
                   label=f'Zona 30 ({ANNO_TRATT_BO})')
        ax.set_title(f'{titolo_out} vs {lbl_ctrl.split("(")[0].strip()}\n'
                     f'δ = {r["coef"]:+.5f}{r["stars"]}, p = {r["pval"]:.4f}', fontsize=9)
        ax.set_xlabel('Anno', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=6)
        ax.legend(fontsize=7); ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
# ====================================================
# DiD — Olbia  (Zona 30: gen 2022)
# ====================================================
ANNO_TRATT_OL = 2022

is_ol = _dj['_key'] == 'olbia'

# Tassi aggregati — mortalità e ferimento
t_ol        = {c: _tasso_agg(is_ol,                                    c) for c in ('morti', 'feriti', 'incidenti')}
t_it_no_ol  = {c: _tasso_agg(~is_ol,                                   c) for c in ('morti', 'feriti', 'incidenti')}
t_sa_no_ol  = {c: _tasso_agg(_dj['_key'].isin(sard_keys) & ~is_ol,     c) for c in ('morti', 'feriti', 'incidenti')}

OUTCOMES = [('morti',     'Mortalità', 'Morti per 1.000 ab.'),
            ('feriti',    'Ferimento', 'Feriti per 1.000 ab.'),
            ('incidenti', 'Incidenti', 'Incidenti per 1.000 ab.')]

_did_results_ol = []

for col, titolo_out, ylabel in OUTCOMES:
    panel_it = build_panel(t_it_no_ol[col], t_ol[col], ANNO_TRATT_OL,
                           'Italia (excl. OL)', 'Olbia')
    panel_sa = build_panel(t_sa_no_ol[col], t_ol[col], ANNO_TRATT_OL,
                           'Sardegna (excl. OL)', 'Olbia')

    res_it, ref_it = stima_did(panel_it, ANNO_TRATT_OL)
    res_sa, ref_sa = stima_did(panel_sa, ANNO_TRATT_OL)

    r_it = extract_did(res_it, f'Olbia vs Italia — {titolo_out}')
    r_sa = extract_did(res_sa, f'Olbia vs Sardegna — {titolo_out}')
    r_it['outcome'] = r_sa['outcome'] = col
    _did_results_ol += [r_it, r_sa]

    # ---- Output numerico ----
    print('=' * 64)
    print(f'DiD — Olbia (Post ≥ {ANNO_TRATT_OL})   [{titolo_out} — tasso per 1.000 ab.]')
    print('=' * 64)
    print(f'\n  vs Italia (anno ref = {ref_it})')
    print_did(r_it)
    print(f'\n  vs Sardegna (anno ref = {ref_sa})')
    print_did(r_sa)
    print()

print('Significatività: *** p<0.01  ** p<0.05  * p<0.1')

# ---- Grafici parallel trends (2×2) ----
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
fig.suptitle('Olbia — Parallel Trends (tassi per 1.000 ab.)',
             fontsize=12, fontweight='bold')

idx = 0
for row_ax, (col, titolo_out, ylabel) in zip(axes, OUTCOMES):
    for ax, ctrl_tasso, lbl_ctrl in zip(
            row_ax,
            [t_it_no_ol[col], t_sa_no_ol[col]],
            ['Italia (esclusa Olbia)', 'Sardegna (esclusa Olbia)']):

        r = _did_results_ol[idx]; idx += 1
        ax.plot(anni, ctrl_tasso,  'o-', color='steelblue', lw=2, ms=4, label=lbl_ctrl)
        ax.plot(anni, t_ol[col],   's-', color='tomato',    lw=2, ms=5, label='Olbia')
        ax.axvline(ANNO_TRATT_OL - 0.5, color='green', ls='--', lw=1.5,
                   label=f'Zona 30 (gen {ANNO_TRATT_OL})')
        ax.set_title(f'{titolo_out} vs {lbl_ctrl.split("(")[0].strip()}\n'
                     f'δ = {r["coef"]:+.5f}{r["stars"]}, p = {r["pval"]:.4f}', fontsize=9)
        ax.set_xlabel('Anno', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_xticks(anni); ax.set_xticklabels(anni, rotation=45, ha='right', fontsize=6)
        ax.legend(fontsize=7); ax.grid(alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
# ====================================================
# Tabella riepilogativa DiD + Coefficient Plot
# ====================================================
all_results = _did_results + _did_results_ol   # 6 Bologna + 6 Olbia = 12 stime

# ---- Tabella ----
rows = []
for r in all_results:
    rows.append({
        'Confronto' : r['label'],
        'δ (DiD)'   : f"{r['coef']:+.5f}{r['stars']}",
        'SE (HC1)'  : f"{r['se']:.5f}",
        't-stat'    : f"{r['tstat']:+.3f}",
        'p-value'   : f"{r['pval']:.4f}",
        'IC 95%'    : f"[{r['ci_low']:+.5f}, {r['ci_high']:+.5f}]",
        'N'         : r['N'],
        'R²'        : f"{r['R2']:.4f}",
    })

df_did = pd.DataFrame(rows)
print('Tabella riepilogativa DiD — mortalità e ferimento (tasso per 1.000 ab.)')
print(df_did.to_string(index=False))
print('\nSignificatività: *** p<0.01  ** p<0.05  * p<0.1')


In [ ]:
# Coefficient Plot DiD — Mortalità
fig, ax = plt.subplots(figsize=(9, 4))

all_results = _did_results + _did_results_ol
subset = [r for r in all_results if r['outcome'] == 'morti']
labels = [r['label'].split(' — ')[0] for r in subset]
coefs  = [r['coef']    for r in subset]
lows   = [r['ci_low']  for r in subset]
highs  = [r['ci_high'] for r in subset]
stars  = [r['stars']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

x_range = max(highs) - min(lows) if highs else 1
for i, (lbl, c, lo, hi, s, col) in enumerate(zip(labels, coefs, lows, highs, stars, cols)):
    ax.plot([lo, hi], [i, i], color=col, lw=2.5)
    ax.plot(c, i, 'o', color=col, ms=8, zorder=5)
    ax.text(max(hi, c) + x_range * 0.02, i, f' {s}', va='center', fontsize=10, color=col)

ax.axvline(0, color='black', lw=1, ls='--')
ax.set_yticks(list(range(len(labels))))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('δ (Morti per 1.000 ab.)', fontsize=10)
ax.set_title('Coefficient Plot DiD — Mortalità  (IC 95%, HC1)', fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Coefficient Plot DiD — Ferimento
fig, ax = plt.subplots(figsize=(9, 4))

all_results = _did_results + _did_results_ol
subset = [r for r in all_results if r['outcome'] == 'feriti']
labels = [r['label'].split(' — ')[0] for r in subset]
coefs  = [r['coef']    for r in subset]
lows   = [r['ci_low']  for r in subset]
highs  = [r['ci_high'] for r in subset]
stars  = [r['stars']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

x_range = max(highs) - min(lows) if highs else 1
for i, (lbl, c, lo, hi, s, col) in enumerate(zip(labels, coefs, lows, highs, stars, cols)):
    ax.plot([lo, hi], [i, i], color=col, lw=2.5)
    ax.plot(c, i, 'o', color=col, ms=8, zorder=5)
    ax.text(max(hi, c) + x_range * 0.02, i, f' {s}', va='center', fontsize=10, color=col)

ax.axvline(0, color='black', lw=1, ls='--')
ax.set_yticks(list(range(len(labels))))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('δ (Feriti per 1.000 ab.)', fontsize=10)
ax.set_title('Coefficient Plot DiD — Ferimento  (IC 95%, HC1)', fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Coefficient Plot DiD — Incidenti
fig, ax = plt.subplots(figsize=(9, 4))

all_results = _did_results + _did_results_ol
subset = [r for r in all_results if r['outcome'] == 'incidenti']
labels = [r['label'].split(' — ')[0] for r in subset]
coefs  = [r['coef']    for r in subset]
lows   = [r['ci_low']  for r in subset]
highs  = [r['ci_high'] for r in subset]
stars  = [r['stars']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

x_range = max(highs) - min(lows) if highs else 1
for i, (lbl, c, lo, hi, s, col) in enumerate(zip(labels, coefs, lows, highs, stars, cols)):
    ax.plot([lo, hi], [i, i], color=col, lw=2.5)
    ax.plot(c, i, 'o', color=col, ms=8, zorder=5)
    ax.text(max(hi, c) + x_range * 0.02, i, f' {s}', va='center', fontsize=10, color=col)

ax.axvline(0, color='black', lw=1, ls='--')
ax.set_yticks(list(range(len(labels))))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('δ (Incidenti per 1.000 ab.)', fontsize=10)
ax.set_title('Coefficient Plot DiD — Incidenti  (IC 95%, HC1)', fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()


## Empirical Bayes Before–After

Correzione del *regression-to-the-mean* tramite il metodo Empirical Bayes (EB).

Il metodo si articola in tre passi:
1. Stima del tasso di riferimento nazionale: $\hat{\lambda} = \sum_i y_i / \sum_i e_i$
2. Peso bayesiano: $z = 1 / (1 + \hat{\lambda}/(\alpha \cdot e_i))$, con $\alpha$ stimato via metodo dei momenti (Hauer 1997)
3. Conteggio atteso EB: $\hat{Y}^{EB} = z \cdot y_{pre} + (1-z) \cdot \hat{\lambda} \cdot e_i$

L'effetto è il rapporto *osservato/atteso EB*, scalato per la durata del periodo post.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ------------------------------------------------------------------
# Funzioni helper Empirical Bayes
# ------------------------------------------------------------------

def expo_col(col_y):
    """Colonna esposizione in _dj: incidenti per morti/feriti; pop. per incidenti."""
    if col_y in ('morti', 'feriti'):
        return lambda a: f'{a}_incidenti'
    else:
        return lambda a: str(a)   # popolazione

def _get_arr(mask, col, anni_list):
    """Array flat (N_comuni × N_anni) per una colonna in _dj."""
    rows = []
    for a in anni_list:
        col_name = col(a) if callable(col) else f'{a}_{col}'
        vals = pd.to_numeric(_dj.loc[mask, col_name], errors='coerce').fillna(0).values
        rows.append(vals)
    return np.concatenate(rows)

def estima_lambda_alpha(mask_ref, col_y, col_e_fn, anni_pre):
    """
    Stima lambda_hat e alpha (Hauer 1997) dal gruppo di controllo
    nel periodo pre-trattamento.
    """
    ys = _get_arr(mask_ref, col_y, anni_pre)
    es = _get_arr(mask_ref, col_e_fn, anni_pre)

    valid = np.isfinite(ys) & np.isfinite(es) & (es > 0) & (ys >= 0)
    ys, es = ys[valid], es[valid]

    lam_hat = ys.sum() / es.sum() if es.sum() > 0 else np.nan

    # Hauer estimator: alpha = sum(e_i^2) / [sum((y_i - lam*e_i)^2 - lam*e_i)]
    resid2  = (ys - lam_hat * es) ** 2 - lam_hat * es
    num     = (es ** 2).sum()
    denom   = resid2.sum()
    alpha   = num / denom if denom > 0 else 1e6   # se nessuna overdispersion → z→1

    return lam_hat, max(alpha, 1e-6)

def eb_analysis(city_key, col_y, mask_ref, ref_label, anni_pre, anni_post):
    """
    Stima EB before–after per una città, un outcome e un gruppo di controllo.
    Restituisce un dizionario con tutti i parametri e l'effetto.
    """
    col_e_fn = expo_col(col_y)
    mask_city = _dj['_key'] == city_key

    lam_hat, alpha = estima_lambda_alpha(mask_ref, col_y, col_e_fn, anni_pre)

    # Conteggi e esposizione pre (città)
    y_pre = sum(
        pd.to_numeric(_dj.loc[mask_city, f'{a}_{col_y}'], errors='coerce').fillna(0).sum()
        for a in anni_pre
    )
    e_pre = sum(
        pd.to_numeric(_dj.loc[mask_city, col_e_fn(a)], errors='coerce').fillna(0).sum()
        for a in anni_pre
    )

    # Conteggi e esposizione post (città)
    y_post = sum(
        pd.to_numeric(_dj.loc[mask_city, f'{a}_{col_y}'], errors='coerce').fillna(0).sum()
        for a in anni_post
    )
    e_post = sum(
        pd.to_numeric(_dj.loc[mask_city, col_e_fn(a)], errors='coerce').fillna(0).sum()
        for a in anni_post
    )

    # Peso bayesiano e conteggio atteso EB (sul periodo pre)
    z     = 1.0 / (1.0 + lam_hat / (alpha * e_pre)) if (alpha * e_pre) > 0 else 0.0
    y_eb  = z * y_pre + (1 - z) * lam_hat * e_pre

    # Tasso EB (per unità di esposizione) → scala al post
    eb_rate      = y_eb / e_pre  if e_pre  > 0 else np.nan
    y_eb_post    = eb_rate * e_post if e_post > 0 else np.nan

    effect_ratio = y_post / y_eb_post if (y_eb_post and y_eb_post > 0) else np.nan
    pct_change   = (effect_ratio - 1) * 100 if not np.isnan(effect_ratio) else np.nan

    return dict(
        ref_label=ref_label, outcome=col_y,
        lam_hat=lam_hat, alpha=alpha, z=z,
        y_pre=y_pre, e_pre=e_pre, y_eb=y_eb, eb_rate=eb_rate,
        y_post=y_post, e_post=e_post, y_eb_post=y_eb_post,
        effect_ratio=effect_ratio, pct_change=pct_change,
        n_pre=len(anni_pre), n_post=len(anni_post),
    )

def print_eb(r):
    print(f"    {r['ref_label']:40s}  "
          f"λ̂={r['lam_hat']:.5f}  α={r['alpha']:.3f}  z={r['z']:.3f}  "
          f"y_post={r['y_post']:.0f}  Ŷ_EB_post={r['y_eb_post']:.2f}  "
          f"ratio={r['effect_ratio']:.3f}  ({r['pct_change']:+.1f}%)")

OUTCOMES_EB = [
    ('morti',     'Mortalità'),
    ('feriti',    'Ferimento'),
    ('incidenti', 'Incidenti'),
]

print("✅ Funzioni Empirical Bayes pronte.")


In [ ]:
# ====================================================
# EB — Bologna  (Zona 30: 2024)
# ====================================================
ANNI_PRE_BO  = list(range(2002, 2024))   # 22 anni pre (2002–2023)
ANNI_POST_BO = [2024]                    # 1 anno post

is_bo     = _dj['_key'] == 'bologna'
ref_it_bo = ~is_bo
ref_er_bo = _dj['_key'].isin(er_keys) & ~is_bo

_eb_results_bo = []

print('=' * 70)
print('Empirical Bayes Before–After — Bologna (Zona 30: 2024)')
print(f'Pre: {ANNI_PRE_BO[0]}–{ANNI_PRE_BO[-1]} ({len(ANNI_PRE_BO)} anni) | '
      f'Post: {ANNI_POST_BO[0]}–{ANNI_POST_BO[-1]} ({len(ANNI_POST_BO)} anno)')
print('=' * 70)

for col, titolo in OUTCOMES_EB:
    r_it = eb_analysis('bologna', col, ref_it_bo, 'Italia (excl. BO)',
                       ANNI_PRE_BO, ANNI_POST_BO)
    r_er = eb_analysis('bologna', col, ref_er_bo, 'Emilia-Romagna (excl. BO)',
                       ANNI_PRE_BO, ANNI_POST_BO)
    for r in (r_it, r_er):
        r['citta']  = 'Bologna'
        r['titolo'] = titolo
        _eb_results_bo.append(r)

    print(f"\n  [{titolo}]  (esposizione: {'incidenti' if col != 'incidenti' else 'popolazione'})")
    print_eb(r_it)
    print_eb(r_er)

print("\n  ratio < 1 → riduzione rispetto all'atteso EB | ratio > 1 → aumento")


In [ ]:
# ====================================================
# EB — Olbia  (Zona 30: gen 2022)
# ====================================================
ANNI_PRE_OL  = list(range(2002, 2022))   # 20 anni pre (2002–2021)
ANNI_POST_OL = list(range(2022, 2025))   # 3 anni post (2022–2024)

is_ol     = _dj['_key'] == 'olbia'
ref_it_ol = ~is_ol
ref_sa_ol = _dj['_key'].isin(sard_keys) & ~is_ol

_eb_results_ol = []

print('=' * 70)
print('Empirical Bayes Before–After — Olbia (Zona 30: gen 2022)')
print(f'Pre: {ANNI_PRE_OL[0]}–{ANNI_PRE_OL[-1]} ({len(ANNI_PRE_OL)} anni) | '
      f'Post: {ANNI_POST_OL[0]}–{ANNI_POST_OL[-1]} ({len(ANNI_POST_OL)} anni)')
print('=' * 70)

for col, titolo in OUTCOMES_EB:
    r_it = eb_analysis('olbia', col, ref_it_ol, 'Italia (excl. OL)',
                       ANNI_PRE_OL, ANNI_POST_OL)
    r_sa = eb_analysis('olbia', col, ref_sa_ol, 'Sardegna (excl. OL)',
                       ANNI_PRE_OL, ANNI_POST_OL)
    for r in (r_it, r_sa):
        r['citta']  = 'Olbia'
        r['titolo'] = titolo
        _eb_results_ol.append(r)

    print(f"\n  [{titolo}]  (esposizione: {'incidenti' if col != 'incidenti' else 'popolazione'})")
    print_eb(r_it)
    print_eb(r_sa)

print("\n  ratio < 1 → riduzione rispetto all'atteso EB | ratio > 1 → aumento")

# ====================================================
# Tabella riepilogativa EB
# ====================================================
all_eb = _eb_results_bo + _eb_results_ol

rows_eb = []
for r in all_eb:
    rows_eb.append({
        'Città'            : r['citta'],
        'Outcome'          : r['titolo'],
        'Riferimento'      : r['ref_label'],
        'λ̂'               : f"{r['lam_hat']:.5f}",
        'α'                : f"{r['alpha']:.2f}",
        'z'                : f"{r['z']:.3f}",
        'Ŷ_EB_post'        : f"{r['y_eb_post']:.1f}",
        'y_post (obs)'     : f"{r['y_post']:.0f}",
        'Ratio (obs/EB)'   : f"{r['effect_ratio']:.3f}",
        'Var. % vs EB'     : f"{r['pct_change']:+.1f}%",
    })

df_eb = pd.DataFrame(rows_eb)
print('\n' + '=' * 70)
print('Tabella riepilogativa — Empirical Bayes Before–After')
print('=' * 70)
print(df_eb.to_string(index=False))


In [ ]:
# EB Effect Ratio — Mortalità
fig, ax = plt.subplots(figsize=(9, 4))

all_eb = _eb_results_bo + _eb_results_ol
subset = [r for r in all_eb if r['outcome'] == 'morti']
labels = [f"{r['citta']} vs {r['ref_label'].split('(')[0].strip()}" for r in subset]
ratios = [r['effect_ratio'] for r in subset]
pcts   = [r['pct_change']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

y_pos = list(range(len(labels)))
ax.barh(y_pos, ratios, color=cols, alpha=0.75, edgecolor='grey', lw=0.5)

x_max = max(max(ratios, default=1), 1.0)
for i, (ratio, pct) in enumerate(zip(ratios, pcts)):
    ax.text(x_max + 0.02, i, f'{pct:+.1f}%', va='center', fontsize=9)

ax.axvline(1, color='black', lw=1.5, ls='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Ratio osservato / atteso EB', fontsize=10)
ax.set_title('EB Effect Ratio — Mortalità\nratio < 1 = riduzione  |  ratio > 1 = aumento',
             fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(left=0, right=x_max + 0.25)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# EB Effect Ratio — Ferimento
fig, ax = plt.subplots(figsize=(9, 4))

all_eb = _eb_results_bo + _eb_results_ol
subset = [r for r in all_eb if r['outcome'] == 'feriti']
labels = [f"{r['citta']} vs {r['ref_label'].split('(')[0].strip()}" for r in subset]
ratios = [r['effect_ratio'] for r in subset]
pcts   = [r['pct_change']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

y_pos = list(range(len(labels)))
ax.barh(y_pos, ratios, color=cols, alpha=0.75, edgecolor='grey', lw=0.5)

x_max = max(max(ratios, default=1), 1.0)
for i, (ratio, pct) in enumerate(zip(ratios, pcts)):
    ax.text(x_max + 0.02, i, f'{pct:+.1f}%', va='center', fontsize=9)

ax.axvline(1, color='black', lw=1.5, ls='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Ratio osservato / atteso EB', fontsize=10)
ax.set_title('EB Effect Ratio — Ferimento\nratio < 1 = riduzione  |  ratio > 1 = aumento',
             fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(left=0, right=x_max + 0.25)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# EB Effect Ratio — Incidenti
fig, ax = plt.subplots(figsize=(9, 4))

all_eb = _eb_results_bo + _eb_results_ol
subset = [r for r in all_eb if r['outcome'] == 'incidenti']
labels = [f"{r['citta']} vs {r['ref_label'].split('(')[0].strip()}" for r in subset]
ratios = [r['effect_ratio'] for r in subset]
pcts   = [r['pct_change']   for r in subset]
cols   = ['tomato', 'tomato', 'steelblue', 'steelblue']

y_pos = list(range(len(labels)))
ax.barh(y_pos, ratios, color=cols, alpha=0.75, edgecolor='grey', lw=0.5)

x_max = max(max(ratios, default=1), 1.0)
for i, (ratio, pct) in enumerate(zip(ratios, pcts)):
    ax.text(x_max + 0.02, i, f'{pct:+.1f}%', va='center', fontsize=9)

ax.axvline(1, color='black', lw=1.5, ls='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Ratio osservato / atteso EB', fontsize=10)
ax.set_title('EB Effect Ratio — Incidenti\nratio < 1 = riduzione  |  ratio > 1 = aumento',
             fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(left=0, right=x_max + 0.25)

bo_patch = mpatches.Patch(color='tomato',    label='Bologna (Zona 30: 2024)')
ol_patch = mpatches.Patch(color='steelblue', label='Olbia (Zona 30: 2022)')
ax.legend(handles=[bo_patch, ol_patch], fontsize=9)
plt.tight_layout()
plt.show()
